In [1]:
import os
import re
from datetime import datetime
import pandas as pd
import numpy as np
from plotnine import *
import baltic as bt
import geopandas as gpd
import geodatasets
from shapely.geometry import Polygon, MultiPolygon
import json
import sys
import subprocess
import glob
import shutil
import importlib
import matplotlib as mpl

In [2]:
def load_tree(nexusPath):
    mytree = bt.loadNexus(nexusPath, tip_regex='\|([0-9]{4}\-[0-9]{2}\-[0-9]{2})', absoluteTime=True, verbose=False) # your tip regex might have to change or you could convert you tree into a newick by deleting everything before the tree string lol
    return(mytree)

In [3]:
# Load tree
nexusPath="./SERAPHIM/nov2021-2024_no-suliformes/NAm_wild-aves_nov2021-2024_no-suliformes_ha_contphyl_2025-08-18_MCC.tree" # your path here
mytree = load_tree(nexusPath)

In [4]:
print(type(mytree))

<class 'baltic.baltic.tree'>


In [5]:
print(mytree.treeStats())


Tree height: 3.325378
Tree length: 696.371663
strictly bifurcating tree
annotations present

Numbers of objects in tree: 10995 (5497 nodes and 5498 leaves)

None


In [6]:
# Add a label for each object in mytree

label_count = 0

for k in mytree.Objects:
    k.label = label_count
    label_count += 1

In [7]:
def return_all_leaves(node):
    leaves[node.label] = node.leaves
    # no_leaves[node.label] = len(node.leaves)
    return leaves

In [13]:
# This loop uses the return_all_leaves function to return a dictionary of all leaves per node

parse_count = 0

for k in mytree.Objects:
    if parse_count < 10:
        if k.is_node():
            node = k
            leaves = {}
            # no_leaves = {}
            return_all_leaves(node)
            parse_count += 1
            #print(leaves)

In [87]:
# Function is meant to make a list of all the nodes in the path to each leaf

def return_path_to_leaf(node, leaf, path_to_leaf):
    for child in node.children:
        # print(f"Child: {child}, has leaves: {hasattr(child, 'leaves')}")
        # print(node.children)
        if child == leaf: # If the child is the leaf, stop parsing
            break
        elif hasattr(child, 'leaves') and leaf in child.leaves:  # Optional optimization: only go down relevant branches
            # path_to_leaf.append(child)
            path_to_leaf.append(child.label)
            return_path_to_leaf(child, leaf, path_to_leaf)
        else:
            pass

In [89]:
parse_count = 0

for k in mytree.Objects:
    if parse_count < 2:
        if k.is_node(): # Parse through each node in the tree
            node = k
            leaves_for_node = []
            leaves_for_node.extend(node.leaves)
            parse_count += 1
            for leaf in leaves_for_node: # print a path to each leaf for each node
                path_to_leaf = []
                return_path_to_leaf(node, leaf, path_to_leaf)
            # print(leaves_for_node[0:3])
            print(path_to_leaf)

[1, 2907, 2908, 4324, 4325, 4326, 4508, 4720, 4738, 4739, 4897, 8959, 8961, 8965, 8975, 8979, 8983, 8991, 8995, 8997, 9003, 9005, 9006, 9008, 9009, 9199, 9200, 9202, 9204, 9206, 9208, 9210, 9214, 9218, 9220, 9222, 9224, 9232, 9233]
[2907, 2908, 4324, 4325, 4326, 4508, 4720, 4738, 4739, 4897, 8959, 8961, 8965, 8975, 8979, 8983, 8991, 8995, 8997, 9003, 9005, 9006, 9008, 9009, 9199, 9200, 9202, 9204, 9206, 9208, 9210, 9214, 9218, 9220, 9222, 9224, 9232, 9233]


In [32]:
# for each node, go 6 months into the future and count how many node and tips have branched from that node

# t = 0.5
# count = 0
# children_count = 0
# child_dict = {}

# for k in mytree.Objects:
#     if count < 10: # only print 10 times
#         key = k.label
#         if k.is_node(): # if an object is a node, then...
#             time = k.length # time is the length of each branch
#             while time < t: # while the time is less than six months
#                 count +=1 # Increase count by one so it knows when to stop counting 
#                 if k.children[0].is_node(): # if the first of K's two children is a node...
#                     children_count += len(k.children) # count the number of childen
#                     if key not in child_dict: # If k is not already in child_dict, add it
#                         child_dict[key] = []
#                     child_dict[key].append(children_count) # Make a dictionary for the number of children in each k
#                     k = k.children[0] # Then name the node child the new k
#                     time += k.length # add the branch length of k to the time (until you get to 6 month)
#                 elif k.children[0].is_leaf(): # If the first child is a 
#                     children_count += len(k.children) # count the number of childen
#                     child_dict = child_dict[key].append(children_count) # Make a dictionary for each
#                 else:
#                     print(f"something unexpected at k {k.label}")

In [ ]:
# Original version of this code (don't want to mess with the original as I try to fix it)

# for k in mytree.Objects:
#     if count >= 10: # only do this 10 times
#         break

#     key = k.label # key is the label (number) of the node
#     time = k.length # time is the length of each branch

#     if k.is_node(): # if k is a leaf:
        
#         # node = k  # use a separate variable to traverse
    
#         while time < t: # while the time is less than six months

#             children_count = len(k.children) # count the number of childen
            
#              # Initialize list if key doesn't exist
#             if key not in child_dict:
#                 child_dict[key] = []
#             child_dict[key].append(children_count) # Make a dictionary for the number of children in each k

            
#             if k.children[0].is_node(): # if the first of K's two children is a node...

#                 children_count += 2
#                 k = k.children[0] # Then name the node child the next node
#                 time += k.length # add the branch length of k to the time (until you get to 6 month)
#                 count += 1 # Increase count by one so it knows when to stop counting
    
#             elif k.children[0].is_leaf(): # If the first child is a leaf

#                 children_count += 1
#                 count += 1 # Increase count by one so it knows when to stop counting
#                 break  # reached leaf, no further traversal
    
#             else:
#                 print(f"something unexpected at node {node.label}")
#                 # break
        
#     elif not k.is_node():
#         child_dict[key].append("not node")

In [26]:
# Second for loop, updated from my attempt by Chat GPT

t = 0.5
count = 0
child_dict = {}

for k in mytree.Objects:
    if count >= 20: # only do this 10 times
        break

    key = k.label # key is the label (number) of the node
    time = k.length # time is the length of each branch

    if k.is_node(): # if k is a leaf:
        
        # node = k  # use a separate variable to traverse
        children_count = 0
    
        while time < t: # while the time is less than six months

            # children_count = len(k.children) # count the number of childen
            
            if k.children[0].is_node(): # if the first of K's two children is a node...

                children_count += 2 # Because a node has two more children...?
                k = k.children[0] # Then name the node child the next node
                time += k.length # add the branch length of k to the time (until you get to 6 month)
                count += 1 # Increase count by one so it knows when to stop counting
    
            elif k.children[0].is_leaf(): # If the first child is a leaf

                children_count += 1 # Because a leaf is one child
                count += 1 # Increase count by one so it knows when to stop counting
                break  # reached leaf, no further traversal
    
            else:
                print(f"something unexpected at node {node.label}")
                # break
        
        # Initialize list if key doesn't exist
        if key not in child_dict:
            child_dict[key] = []
        child_dict[key].append(children_count) # Make a dictionary for the number of children in each k

    elif not k.is_node():
        child_dict[key].append("not node")

In [27]:
print(child_dict)

{0: [5], 1: [6], 2: [5], 3: [9], 4: [9], 5: [7]}


In [ ]:
# Eventually will want to ask where each node/tip are

In [ ]:
# Function: calculate distance 

def calc_dist(j):
    # Establish latitude and longitude values for the starting node and each of the two children
    node_lat = j.traits["location1_median"]
    node_long = j.traits["location2_median"]
    child1_lat = j.children[0].traits["location1_median"]
    child1_long = j.children[0].traits["location2_median"]
    child2_lat = j.children[1].traits["location1_median"]
    child2_long = j.children[1].traits["location2_median"]

    # Establish the coordinates
    start = (node_lat, node_long)
    child1_end = (child1_lat, child1_long)
    child2_end = (child2_lat, child2_long)

    # Calculate distance from node to each child
    child1_dist = geodesic(start, child1_end, ellipsoid='GRS-80').kilometers
    child2_dist = geodesic(start, child2_end, ellipsoid='GRS-80').kilometers

    # Average the distance
    avg_distance = (child1_dist + child2_dist) / 2

    # Set the distance as a new trait
    j.distance = avg_distance
    
    return(j.distance)

In [ ]:
# Function: calculate direction

def calc_direction(j):
    # Establish latitude and longitude values for the starting node and each of the two children
    node_lat = j.traits["location1_median"]
    node_long = j.traits["location2_median"]
    child1_lat = j.children[0].traits["location1_median"]
    child1_long = j.children[0].traits["location2_median"]
    child2_lat = j.children[1].traits["location1_median"]
    child2_long = j.children[1].traits["location2_median"]

    # Calculate change in latitude
    child1_changeLat = child1_lat - node_lat
    child2_changeLat = child2_lat - node_lat
    avg_changeLat = (child1_changeLat + child2_changeLat) / 2

    # Calculate change in longitude
    child1_changeLong = child1_long - node_long
    child2_changeLong = child2_long - node_long
    avg_changeLong = (child1_changeLong + child2_changeLong) / 2

    # Set these as new traits
    j.changeLat = avg_changeLat
    j.changeLon = avg_changeLong

    return(j.changeLat, j.changeLon)